# Model development

Final notebook that assumes all insights from `EDA.ipynb` and `Baseline.ipynb` to develop production model

## Download data

In [ ]:
import pandas as pd 
import numpy as np
import sklearn

pd.set_option('display.max_columns', 200)
np.random.seed(42)

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)
df = pd.read_csv(path)
df.head()

## Data preparation

In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

to_category_columns = (
    [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "Churn"
        ])
    
for column in to_category_columns:
    df[column] = df[column].astype("category")
    
df["SeniorCitizen"] = df["SeniorCitizen"] == 1
df["Churn"] = df["Churn"] == "Yes"
    
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')

df = df.dropna()
df["TotalCharges"].isna().sum()


print("Basic data preparation is done")
print(df.shape)


### Removing columns

In [ ]:
# Drop customerId column as never significant
df = df.drop(columns=["customerID", "gender", "MultipleLines", "StreamingTV", "StreamingMovies"])
print("Dropping not significant columns")
print(df.shape)

print("Apply OHE for data")
df_ohe = pd.get_dummies(df)
print(df_ohe.shape)

## Model building

I will use a GBDT approach as one of the most powerful in `Baseline.ipynb`

In [ ]:
%pip install lightgbm -q

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

churn_ratio = df_ohe.groupby("Churn")["Churn"].count()

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

minor_base_scale = churn_ratio.iloc[0] / churn_ratio.iloc[1]
lgbm = lgb.LGBMClassifier(scale_pos_weight=minor_base_scale, verbose=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(lgbm, X, y, cv=skf)
y_proba_cv = cross_val_predict(lgbm, X, y, cv=skf, method="predict_proba")[:, 1]
auc_scores = cross_val_score(lgbm, X, y, cv=skf, scoring='average_precision')
print(classification_report(y, y_pred_cv))
print(f"PR-AUC: {auc_scores.mean():.4f} +/- {auc_scores.std():.4f} (var: {auc_scores.var():.4f})")

### Model tuning

Lets tune model with optuna.

I will tune model both on PR-AUC score and recall metrics. At the final I will compare models via PR-AUC curves with cost analysis

In [ ]:
%pip install optuna -q

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from matplotlib.ticker import FuncFormatter

# Objective func to train models
def objective(trial, metric_func, threshold=False):
    data, target = X, y
    train_x, valid_x, train_y, valid_y = train_test_split(data, target, test_size=0.25, random_state=42)
    dtrain = lgb.Dataset(train_x, label=train_y)

    param = {
        "objective": "binary",
        "metric": "binary_logloss",
        "verbosity": -1,
        "seed": 42,
        "boosting_type": "gbdt",
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }

    gbm = lgb.train(param, dtrain)
    preds = gbm.predict(valid_x)
    if threshold:
        preds = (preds > 0.5).astype(int)
    return metric_func(valid_y, preds)


FN_cost = 997.94
FP_cost = 89.33
retain_p = 0.45


def compute_cv_fold_costs(study, X, y, n_splits=5):
    best_params = study.best_trial.params
    best_params.update({"objective": "binary", "metric": "binary_logloss", "verbosity": -1, "boosting_type": "gbdt"})

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_money_saved = []
    fold_no_model_costs = []
    fold_best_costs = []
    all_oof_proba = np.zeros(len(y))

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        lgbm_fold = lgb.LGBMClassifier(**best_params)
        lgbm_fold.fit(X_train, y_train)
        y_proba_fold = lgbm_fold.predict_proba(X_test)[:, 1]
        all_oof_proba[test_idx] = y_proba_fold

        precision, recall, thresholds = precision_recall_curve(y_test, y_proba_fold)

        P_fold = y_test.sum()
        TP = recall[:-1] * P_fold
        FN = P_fold - TP
        FP = TP * (1 / precision[:-1] - 1)
        total_costs = FN * FN_cost + FP_cost * FP + (FP_cost + (1 - retain_p) * FN_cost) * TP

        best_idx = np.argmin(total_costs)
        no_model_cost = P_fold * FN_cost
        fold_no_model_costs.append(no_model_cost)
        fold_best_costs.append(total_costs[best_idx])
        fold_money_saved.append(no_model_cost - total_costs[best_idx])

    return np.array(fold_money_saved), np.array(fold_no_model_costs), np.array(fold_best_costs), all_oof_proba


def compute_ci(fold_values):
    from scipy import stats
    n = len(fold_values)
    std = fold_values.std(ddof=1)
    se_total = np.sqrt(n) * std
    ci95 = stats.t.ppf(0.975, df=n - 1) * se_total
    return std, ci95


def plot_cost_curve(thresholds, total_costs, no_model_cost, best_threshold, metric, retain_p=0.45):
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, total_costs, label=f"Retain chance: {retain_p*100:.0f}%", color="olive", linestyle="-.")
    plt.plot(thresholds, [no_model_cost] * len(thresholds), label="No model", color="black", linestyle="--")
    plt.axvline(best_threshold, color="red", linestyle=":", label=f"Best threshold: {best_threshold:.3f}")
    plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x:,.0f}"))
    plt.xlabel("Threshold")
    plt.ylabel("Total Cost ($)")
    plt.title(f"Total Cost curve with optimized {metric} (45% Retention)")
    plt.legend()
    plt.show()


def plot_model_threshold_cost_curve(study, metric: str):
    fold_money_saved, fold_no_model_costs, fold_best_costs, all_oof_proba = compute_cv_fold_costs(study, X, y)

    saved_std, saved_ci = compute_ci(fold_money_saved)
    no_model_std, no_model_ci = compute_ci(fold_no_model_costs)
    best_cost_std, best_cost_ci = compute_ci(fold_best_costs)

    P = y.sum()
    precision, recall, thresholds = precision_recall_curve(y, all_oof_proba)
    TP = recall[:-1] * P
    FN = P - TP
    FP = TP * (1 / precision[:-1] - 1)
    total_costs = FN * FN_cost + FP_cost * FP + (FP_cost + (1 - retain_p) * FN_cost) * TP
    best_idx = np.argmin(total_costs)
    best_threshold = thresholds[best_idx]
    best_cost = total_costs[best_idx]
    no_model_cost = P * FN_cost
    money_saved = no_model_cost - best_cost

    plot_cost_curve(thresholds, total_costs, no_model_cost, best_threshold, metric)

    print(f"Best threshold:  {best_threshold:.4f}")
    print(f"No-model cost:   ${no_model_cost:,.0f} ± ${no_model_std:,.0f} (95% CI: [${no_model_cost - no_model_ci:,.0f}, ${no_model_cost + no_model_ci:,.0f}])")
    print(f"Min model cost:  ${best_cost:,.0f} ± ${best_cost_std:,.0f} (95% CI: [${best_cost - best_cost_ci:,.0f}, ${best_cost + best_cost_ci:,.0f}])")
    print(f"Money saved:     ${money_saved:,.0f} ± ${saved_std:,.0f} (95% CI: [${money_saved - saved_ci:,.0f}, ${money_saved + saved_ci:,.0f}])")
    print(f"Money saved %:   {money_saved/no_model_cost*100:.2f}% ± {saved_std/no_model_cost*100:.2f}%")

### Optimizing PR-AUC score

In [ ]:
from functools import partial

study_auc = optuna.create_study(direction="maximize")
auc_objective = partial(objective, metric_func = sklearn.metrics.average_precision_score)

study_auc.optimize(auc_objective, n_trials=100)

print("Number of finished trials: {}".format(len(study_auc.trials)))

print("Best trial:")
trial = ststudy_aucudy.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [ ]:
plot_model_threshold_cost_curve(study_auc, "PR-AUC")

### Optimizing Recall

In [ ]:
from functools import partial

study_recall = optuna.create_study(direction="maximize")
recall_objective = partial(objective, metric_func = sklearn.metrics.recall_score, threshold=True)

study_recall.optimize(recall_objective, n_trials=100)

print("Number of finished trials: {}".format(len(study_recall.trials)))

print("Best trial:")
trial = study_recall.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [ ]:
plot_model_threshold_cost_curve(study_recall, "Recall")

The bottom line is that we can use both optimizations: AUC or Recall as they give not statistically different results.

Current model save $\approx{22.5}$ % of business loses on 2 year interval ($400,000$ $)

## Model interpretation

In [ ]:
%pip install shap -q

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X)